# R2 Additional Experiments — IP&M reject-and-resubmit (round 2)

Companion to `ids_llm_nebius_v8.ipynb` and the repo at
`https://anonymous.4open.science/r/ai_for_ids`. Three cells, mapped to the round-2 reviews:

| Cell | Reviewer point | Needs API? |
|---|---|---|
| **A1-AGG** — aggregate the new 5-seed N-BaIoT + CIC-UNSW runs into paper tables (Table `t:multiseed-extension`, `t:unsw-rareclass`) | R1: "multi-seed coverage is incomplete", "gains not generalisable" | No |
| **E16-COST** — alarm-level cost-sensitive analysis (Table `t:alarm-cost`) | R1: "connect macro-F1 to real operational costs" | No |
| **E15-FB** — feature-budget sensitivity sweep K ∈ {6, 12, 18, 24} (Table `t:e15-feature-budget`) | R1: "fixed K=12 requires ... sensitivity analysis" | **Yes (Nebius)** |

**How to run E15-FB:** open `ids_llm_nebius_v8.ipynb`, execute the setup cells 0–22
(config, Nebius client, data load for CIC-IDS2017 grouped, helpers, RF anchor), then
paste/run the E15-FB cell below in the same kernel. It reuses `data`,
`select_top_features`, `build_icl_test_split`, `run_icl`, `metrics_from_result`,
`tabular_baseline`, `PRIMARY_MODEL`, `bootstrap_ci`, `BASE` from that notebook.
A1-AGG and E16-COST are standalone (pure pandas/scipy on committed CSVs).

**Bundled outputs.** All three cells' outputs are committed: the multi-seed
extension tables in `data_v8/multiseed_extension/`, the cost analysis in
`data_v8/e16_cost/`, and the feature-budget sweep in `data_v8/e15_feature_budget/`
(E15-FB writes to `./ids_llm_v8/e15_feature_budget/` at runtime; the bundled copy
lives under `data_v8/` alongside the other revision outputs). The raw five-seed
runs the A1-AGG cell aggregates are in `multiseed_runs/{n_baiot,cic_unsw}_V*/`,
produced by the headless runner `multiseed_validation.py`. No cell needs to be
re-run to inspect any round-2 table.

In [ ]:
# ============================================================
# A1-AGG) Aggregate the new five-seed N-BaIoT + CIC-UNSW runs
# ------------------------------------------------------------
# Input : multiseed_runs/{n_baiot_V1..V4, cic_unsw_V1..V2}/
#           {model}_per_seed.csv  and  {model}_summary.json
#         (committed to the repo root, same layout as the
#          existing cic_ids2017_V* folders)
# Output: data_v8/multiseed_extension/
#           multiseed_extension_summary.csv
#           table_multiseed_extension.tex   (Table t:multiseed-extension rows)
#           table_unsw_rareclass.tex        (Table t:unsw-rareclass)
# Stats : two-sided Mann-Whitney U vs Random Forest on the 5
#         seed-level values, Bonferroni-corrected over K=4
#         foundation-model comparisons per cell — identical to the
#         CIC-IDS2017 protocol of Section 5.3.
# ============================================================
import os, json, glob
import pandas as pd
from scipy.stats import mannwhitneyu

MS_DIR  = "multiseed_runs"                      # repo-relative; adjust if needed
OUT_DIR = "data_v8/multiseed_extension"; os.makedirs(OUT_DIR, exist_ok=True)

CELLS = ["n_baiot_V1","n_baiot_V2","n_baiot_V3","n_baiot_V4",
         "cic_unsw_V1","cic_unsw_V2"]
DISP  = {"random_forest":"Random Forest","tabpfn":"TabPFN",
         "tabpfn_2.5":"TabPFN~2.5","tabicl":"TabICL","tabstar":"TabSTAR"}
NAME  = {"n_baiot":"N-BaIoT","cic_unsw":"CIC-UNSW"}

rows, tex = [], []
for cell in CELLS:
    ds, var = cell.rsplit("_", 1)
    rf = pd.read_csv(f"{MS_DIR}/{cell}/random_forest_per_seed.csv")
    for m in ["random_forest","tabpfn","tabpfn_2.5","tabicl","tabstar"]:
        js = json.load(open(f"{MS_DIR}/{cell}/{m}_summary.json"))
        rec = dict(cell=cell, model=m,
                   acc=js["accuracy_mean"], acc_lo=js["accuracy_lo95"], acc_hi=js["accuracy_hi95"],
                   mf1=js["macro_f1_mean"], mf1_lo=js["macro_f1_lo95"], mf1_hi=js["macro_f1_hi95"],
                   p_acc_bonf=None, p_mf1_bonf=None)
        if m != "random_forest":
            fm = pd.read_csv(f"{MS_DIR}/{cell}/{m}_per_seed.csv")
            rec["p_acc_bonf"] = min(mannwhitneyu(fm.accuracy, rf.accuracy).pvalue*4, 1.0)
            rec["p_mf1_bonf"] = min(mannwhitneyu(fm.macro_f1, rf.macro_f1).pvalue*4, 1.0)
        rows.append(rec)
        sa = r"\,$\bigstar$" if (rec["p_acc_bonf"] or 1) < 0.05 else ""
        sf = r"\,$\bigstar$" if (rec["p_mf1_bonf"] or 1) < 0.05 else ""
        lead = (r"    \multirow{5}{*}{%s %s} & " % (NAME[ds], f"$V_{var[1]}$")) \
               if m == "random_forest" else "     & "
        tex.append(f"{lead}{DISP[m]} & ${rec['acc']:.4f}$ $[{rec['acc_lo']:.4f}, {rec['acc_hi']:.4f}]${sa}"
                   f" & ${rec['mf1']:.4f}$ $[{rec['mf1_lo']:.4f}, {rec['mf1_hi']:.4f}]${sf} \\\\")
    tex.append(r"    \hline")

pd.DataFrame(rows).to_csv(f"{OUT_DIR}/multiseed_extension_summary.csv", index=False)
open(f"{OUT_DIR}/table_multiseed_extension.tex","w").write("\n".join(tex))
print(f"[A1-AGG] wrote {OUT_DIR}/multiseed_extension_summary.csv and table_multiseed_extension.tex")

# --- CIC-UNSW rare-class table (classes RF collapses on) --------------------
RARE = ["Analysis","Backdoor","DoS","Fuzzers","Worms"]
lines = []
for cell in ["cic_unsw_V1","cic_unsw_V2"]:
    lines.append(r"    \multicolumn{6}{l}{\emph{CIC-UNSW $V_%s$}} \\" % cell[-1])
    per = {m: json.load(open(f"{MS_DIR}/{cell}/{m}_summary.json"))["per_class_f1"]
           for m in ["random_forest","tabpfn","tabpfn_2.5","tabicl","tabstar"]}
    for c in RARE:
        v = [per[m].get(c, {}).get("f1_mean", float("nan"))
             for m in ["random_forest","tabpfn","tabpfn_2.5","tabicl","tabstar"]]
        lines.append(f"    {c:<8} & " + " & ".join(f"${x:.3f}$" for x in v) + r" \\")
    lines.append(r"    \hline")
open(f"{OUT_DIR}/table_unsw_rareclass.tex","w").write("\n".join(lines))
print(f"[A1-AGG] wrote {OUT_DIR}/table_unsw_rareclass.tex")
pd.DataFrame(rows).head(10)

[A1-AGG] wrote data_v8/multiseed_extension/multiseed_extension_summary.csv and table_multiseed_extension.tex
[A1-AGG] wrote data_v8/multiseed_extension/table_unsw_rareclass.tex


,cell,model,acc,acc_lo,acc_hi,mf1,mf1_lo,mf1_hi,p_acc_bonf,p_mf1_bonf
0,n_baiot_V1,random_forest,0.995349,0.994877,0.995854,0.983441,0.981088,0.986274,NaN,NaN
1,n_baiot_V1,tabpfn,0.956817,0.955890,0.958546,0.874938,0.873751,0.876703,0.031746,0.031746
2,n_baiot_V1,tabpfn_2.5,0.949746,0.934315,0.958717,0.865946,0.844000,0.878235,0.031746,0.031746
3,n_baiot_V1,tabicl,0.957238,0.955477,0.958998,0.876384,0.874627,0.878140,0.031746,0.031746
4,n_baiot_V1,tabstar,0.990673,0.984594,0.995042,0.987194,0.981635,0.991144,0.888889,0.888889
5,n_baiot_V2,random_forest,0.921230,0.917060,0.925399,0.919033,0.915599,0.922467,NaN,NaN
6,n_baiot_V2,tabpfn,0.953687,0.952178,0.955816,0.870621,0.868483,0.873479,0.031746,0.031746
7,n_baiot_V2,tabpfn_2.5,0.957461,0.934476,0.980226,0.888337,0.843408,0.945241,0.222222,0.603175
8,n_baiot_V2,tabicl,0.954537,0.953093,0.955917,0.872952,0.871581,0.874323,0.031746,0.031746
9,n_baiot_V2,tabstar,0.963335,0.952389,0.972028,0.964084,0.957795,0.970442,0.031746,0.031746


In [ ]:
# ============================================================
# E16-COST) Alarm-level cost-sensitive analysis  (no API calls)
# ------------------------------------------------------------
# Reviewer #1: "the paper places strong emphasis on macro-F1 ...
# but does not sufficiently connect these improvements to real
# operational costs. In practical IDS systems, false positives and
# false negatives have asymmetric and often severe consequences."
#
# Decision-level binarisation:
#   miss (FN)        = attack flow predicted BENIGN
#   false alarm (FP) = benign flow predicted as any attack class
#   attack->other-attack misrouting still raises an alarm.
# Both follow exactly from the BENIGN class's precision/recall/support:
#   FN = TP_B * (1-P_B)/P_B    (false positives OF the benign class)
#   FP = n_B * (1-R_B)         (false negatives OF the benign class)
# Expected cost/flow = C_FP*FP + C_FN*FN ; report counts per 100k flows
# and the crossover ratio rho* = C_FN/C_FP vs the accuracy leader.
#
# Axis 1 inputs (5-seed): data_v8/rf_control_and_xgb/cic_ids2017_V1/
#   {rf_pruned_fulltrain, rf_strong_balanced, xgboost}_perclass.csv
# TabSTAR (single-seed): BENIGN P/R from paper Table 16.
# Axis 2 inputs (single-seed): data_v8/reviewer_response/
#   e14_rareclass_deepdive/cm_{llm,rf}.csv
# Output: data_v8/e16_cost/ {e16_axis1_cost.csv, e16_axis2_cost.csv,
#   table_alarm_cost.tex, e16_cost_curves.png}
# ============================================================
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt

A1  = "data_v8/rf_control_and_xgb/cic_ids2017_V1"
A2  = "data_v8/reviewer_response/e14_rareclass_deepdive"
OUT = "data_v8/e16_cost"; os.makedirs(OUT, exist_ok=True)

def alarm_decomp(perclass_csv):
    """5-seed mean (miss, false-alarm) per 100k flows from BENIGN P/R/support."""
    df = pd.read_csv(perclass_csv); fn, fp = [], []
    for _, g in df.groupby("seed"):
        b  = g[g.cls == "BENIGN"].iloc[0]
        n  = g.support.sum(); tp = b.recall * b.support
        fn.append(tp * (1 - b.precision) / b.precision / n * 1e5)
        fp.append(b.support * (1 - b.recall) / n * 1e5)
    n_att = n - g[g.cls == "BENIGN"].support.iloc[0]
    return np.mean(fn), np.mean(fp), n, n_att

rows = []
for name, f in [("RF pruned (full)", "rf_pruned_fulltrain_perclass.csv"),
                ("RF strong (bal.)", "rf_strong_balanced_perclass.csv"),
                ("XGBoost (full)",   "xgboost_perclass.csv")]:
    FN, FP, n, na = alarm_decomp(f"{A1}/{f}")
    rows.append(dict(model=name, miss_per100k=FN, fa_per100k=FP,
                     miss_rate=FN / (na / n * 1e5), fa_rate=FP / ((n - na) / n * 1e5)))

# TabSTAR single-seed (paper Table 16: BENIGN P=0.9995, R=0.9763); supports
# from any seed of the control files (test partition is fixed across seeds).
g   = pd.read_csv(f"{A1}/rf_pruned_fulltrain_perclass.csv"); g = g[g.seed == 0]
nB  = g[g.cls == "BENIGN"].support.iloc[0]; n = g.support.sum()
P, R = 0.9995, 0.9763; tp = R * nB
rows.append(dict(model="TabSTAR (single-seed)",
                 miss_per100k=tp * (1 - P) / P / n * 1e5,
                 fa_per100k=nB * (1 - R) / n * 1e5,
                 miss_rate=(tp * (1 - P) / P) / (n - nB), fa_rate=1 - R))
ax1 = pd.DataFrame(rows)

# crossover rho* vs the aggregate-accuracy leader (pruned RF)
lead = ax1.iloc[0]
ax1["rho_star_vs_leader"] = [
    np.nan if i == 0 else
    (lead.fa_per100k - r.fa_per100k) / (r.miss_per100k - lead.miss_per100k)
    for i, r in ax1.iterrows()]
ax1.to_csv(f"{OUT}/e16_axis1_cost.csv", index=False)
print(ax1.round(3).to_string(index=False))

# --- Axis 2 (balanced protocol, single-seed confusion matrices) -------------
a2 = []
for name, f in [("Qwen3-235B-A22B", "cm_llm.csv"), ("Random Forest", "cm_rf.csv")]:
    cm  = pd.read_csv(f"{A2}/{f}", index_col=0)
    att = [c for c in cm.index if c != "BENIGN"]
    a2.append(dict(model=name,
                   missed=int(cm.loc[att, "BENIGN"].sum()),
                   miss_rate=cm.loc[att, "BENIGN"].sum() / cm.loc[att].values.sum(),
                   false_alarms=int(cm.loc["BENIGN", att].sum()),
                   fa_rate=cm.loc["BENIGN", att].sum() / cm.loc["BENIGN"].values.sum()))
ax2 = pd.DataFrame(a2); ax2.to_csv(f"{OUT}/e16_axis2_cost.csv", index=False)
print("\n", ax2.round(3).to_string(index=False))

# --- expected-cost curves + LaTeX table -------------------------------------
rho = np.logspace(-2, 3, 200)
plt.figure(figsize=(6, 4))
for _, r in ax1.iterrows():
    plt.plot(rho, r.fa_per100k + rho * r.miss_per100k, label=r.model)
plt.xscale("log"); plt.yscale("log")
plt.xlabel(r"cost ratio $\rho = C_{FN}/C_{FP}$")
plt.ylabel("expected cost per 100k flows ($C_{FP}=1$)")
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f"{OUT}/e16_cost_curves.png", dpi=200); plt.close()

tex = [r"    \hline"]
for _, r in ax1.iterrows():
    rs = "--" if np.isnan(r.rho_star_vs_leader) else f"${r.rho_star_vs_leader:.2f}$"
    tex.append(f"    {r.model} & ${r.miss_per100k:.1f}$ \\;(${r.miss_rate*100:.2f}\\%$ of attacks)"
               f" & ${r.fa_per100k:.1f}$ & {rs} \\\\")
tex.append(r"    \hline")
open(f"{OUT}/table_alarm_cost.tex", "w").write("\n".join(tex))
print(f"\n[E16-COST] wrote {OUT}/table_alarm_cost.tex and e16_cost_curves.png")

                model  miss_per100k  fa_per100k  miss_rate  fa_rate  rho_star_vs_leader
     RF pruned (full)       675.357      22.284      0.045    0.000                 NaN
     RF strong (bal.)        43.194    1759.262      0.003    0.021               2.748
       XGBoost (full)        44.508      48.332      0.003    0.001               0.041
TabSTAR (single-seed)        41.476    2012.671      0.003    0.024               3.140

           model  missed  miss_rate  false_alarms  fa_rate
Qwen3-235B-A22B       7      0.021            24     0.48
  Random Forest      14      0.042            36     0.72

[E16-COST] wrote data_v8/e16_cost/table_alarm_cost.tex and e16_cost_curves.png


In [ ]:
# ============================================================
# E15-PREREQ) Standalone setup for the E15-FB sweep
# ------------------------------------------------------------
# Condenses cells 0-22 of ids_llm_nebius_v8.ipynb to exactly the
# objects E15-FB needs: data, select_top_features,
# build_icl_test_split, run_icl, metrics_from_result,
# tabular_baseline, PRIMARY_MODEL, bootstrap_ci, BASE.
# Protocol constants identical to the v8 'paper' scale:
#   grouped CIC-IDS2017 (dhoogla mirror), <=50k rows/class,
#   MI feature selection, ICL 5/class, test 50/class,
#   raw-json prompts, n_vote=3 @ T=0.7, RF n_estimators=200.
# Secrets needed: NEBIUS_API_KEY, KAGGLE_USERNAME, KAGGLE_KEY.
# ============================================================
!pip install -q openai==1.55.* tqdm pandas scikit-learn scipy matplotlib kagglehub

import os, json, random, hashlib, time, re, gc, glob
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple, Optional
from collections import Counter
import numpy as np, pandas as pd
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings("ignore")

# ---- 1) Nebius client (with the cell-3 key sanitization folded in) --------
from openai import OpenAI
try:
    from google.colab import userdata
    _raw = userdata.get("NEBIUS_API_KEY")
except Exception:
    _raw = os.environ.get("NEBIUS_API_KEY", "")
_JUNK = {"\ufeff", "\u200b", "\u200c", "\u200d", "\u2060", "\u00a0"}
NEBIUS_KEY = "".join(ch for ch in _raw if ch not in _JUNK).strip().strip("'\"").strip("\u2018\u2019\u201c\u201d").strip()
assert NEBIUS_KEY and all(ord(c) < 128 for c in NEBIUS_KEY), "Set a clean NEBIUS_API_KEY in Colab Secrets or env."
client = OpenAI(base_url="https://api.studio.nebius.com/v1/", api_key=NEBIUS_KEY)

PRIMARY_MODEL  = "Qwen3-235B-A22B"
MODEL_REGISTRY = {"Qwen3-235B-A22B": "Qwen/Qwen3-235B-A22B-Instruct-2507"}

# ---- 2) utilities ----------------------------------------------------------
def set_seed(s): random.seed(s); np.random.seed(s)
def ensure_dir(p): os.makedirs(p, exist_ok=True); return p
def _sha1(s): return hashlib.sha1(s.encode()).hexdigest()[:12]

def bootstrap_ci(x, n_boot: int = 5000, ci: float = 0.95, seed: int = 0):
    x = np.asarray(x, dtype=float)
    if len(x) == 0: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    lo, hi = np.percentile(means, [100*(1-ci)/2, 100*(1+ci)/2])
    return float(np.mean(x)), float(lo), float(hi)

BASE = ensure_dir("./ids_llm_v8")

# ---- 3) NebiusLLM wrapper (sequential n_vote, retries) ---------------------
class NebiusLLM:
    def __init__(self, client, model_id, name, temperature=0.7, max_tokens=32, cache_enabled=True):
        self.client, self.model_id, self.name = client, model_id, name
        self.temperature, self.max_tokens = temperature, max_tokens
        self.cache_enabled = cache_enabled and (temperature == 0.0)
        self._cache = {}
        self.stats = {"name": name, "calls": 0, "cached": 0, "errors": 0,
                      "prompt_tokens": 0, "completion_tokens": 0, "wall_seconds": 0.0}
    def _key(self, system, prompt):
        return _sha1(f"{self.model_id}|t={self.temperature}|sys={system}|p={prompt}")
    def _one_call(self, prompt, system=None):
        key = self._key(system, prompt)
        if self.cache_enabled and key in self._cache:
            self.stats["cached"] += 1; return self._cache[key]
        msgs = ([{"role": "system", "content": system}] if system else []) + \
               [{"role": "user", "content": prompt}]
        for attempt in range(4):
            try:
                t0 = time.time()
                resp = self.client.chat.completions.create(
                    model=self.model_id, messages=msgs,
                    temperature=self.temperature, max_tokens=self.max_tokens, n=1)
                self.stats["calls"] += 1
                self.stats["wall_seconds"] += time.time() - t0
                text = resp.choices[0].message.content.strip()
                if getattr(resp, "usage", None):
                    self.stats["prompt_tokens"]     += resp.usage.prompt_tokens
                    self.stats["completion_tokens"] += resp.usage.completion_tokens
                if self.cache_enabled: self._cache[key] = text
                return text
            except Exception:
                self.stats["errors"] += 1
                if attempt == 3: return ""
                time.sleep(min(2**attempt + random.random(), 12))
    def chat(self, prompt, system=None, n=1):
        return [self._one_call(prompt, system=system) for _ in range(n)]

# ---- 4) data: grouped CIC-IDS2017 via kagglehub ---------------------------
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        if not os.environ.get(k): os.environ[k] = userdata.get(k)
except Exception:
    pass
assert os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"), \
    "Set KAGGLE_USERNAME and KAGGLE_KEY (Colab Secrets or env)."

CIC_CLASS_MAPPING = {
    'BENIGN':'BENIGN', 'Benign':'BENIGN', 'DDoS':'DDoS',
    'DoS Hulk':'DoS', 'DoS GoldenEye':'DoS', 'DoS slowloris':'DoS', 'DoS Slowhttptest':'DoS',
    'PortScan':'Port Scan', 'Portscan':'Port Scan',
    'FTP-Patator':'Brute Force', 'SSH-Patator':'Brute Force', 'Bruteforce':'Brute Force',
    'Bot':'Bot',
    'Web Attack \x96 Brute Force':'Web Attack', 'Web Attack \x96 Sql Injection':'Web Attack',
    'Web Attack \x96 XSS':'Web Attack', 'Web Attack  Brute Force':'Web Attack',
    'Web Attack  Sql Injection':'Web Attack', 'Web Attack  XSS':'Web Attack',
    'Web Attack':'Web Attack', 'Web Attacks':'Web Attack',
    'Infiltration':'Infiltration', 'Heartbleed':'Heartbleed',
}

def _basic_clean(df):
    df = df.copy()
    df.drop_duplicates(inplace=True)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    for col in ("Flow_Bytes/s", "Flow_Packets/s"):
        if col in df.columns: df[col] = df[col].fillna(df[col].median())
    df.dropna(inplace=True)
    for col in df.columns:
        t = df[col].dtype
        if t == object: continue
        cmin, cmax = df[col].min(), df[col].max()
        if "float" in str(t) and cmin > np.finfo(np.float32).min and cmax < np.finfo(np.float32).max:
            df[col] = df[col].astype(np.float32)
        elif "int" in str(t) and cmin > np.iinfo(np.int32).min and cmax < np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)
    n_unique = df.nunique()
    return df[n_unique[n_unique > 1].index]

import kagglehub
folder = kagglehub.dataset_download("dhoogla/cicids2017")
parquets = sorted(glob.glob(os.path.join(folder, "**", "*.parquet"), recursive=True))
if parquets:
    raw = pd.concat([pd.read_parquet(p) for p in parquets], ignore_index=True)
else:
    csvs = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
    raw = pd.concat([pd.read_csv(c, encoding="latin-1", low_memory=False) for c in csvs], ignore_index=True)
raw.columns = [c.strip().replace(" ", "_") for c in raw.columns]

# grouped preprocessing (v8 preprocess_cic, CLASS_GROUPING='grouped')
if "ClassLabel" in raw.columns:
    raw = raw.drop(columns=["Label"], errors="ignore").rename(columns={"ClassLabel": "Label"})
    already_mapped = True
else:
    already_mapped = False
raw["Label"] = raw["Label"].astype(str).str.replace(r"[^\x00-\x7F]+", "", regex=True).str.strip()
data = _basic_clean(raw)
if not already_mapped:
    data["Label"] = data["Label"].map(lambda x: CIC_CLASS_MAPPING.get(x, x))
data["Label"] = data["Label"].replace({"Benign": "BENIGN"})
del raw; gc.collect()

SUBSAMPLE_PER_CLASS = 50_000
if data["Label"].value_counts().max() > SUBSAMPLE_PER_CLASS:
    data = data.groupby("Label", group_keys=False).apply(
        lambda g: g.sample(min(len(g), SUBSAMPLE_PER_CLASS), random_state=0)
    ).reset_index(drop=True)
print(f"data: {data.shape}\n{data['Label'].value_counts()}")

# ---- 5) feature selection + samplers (paper-scale defaults baked in) ------
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

def select_top_features(df, k=12, sample=50_000, seed=0):
    rng = np.random.default_rng(seed)
    sub = df.iloc[rng.choice(len(df), size=sample, replace=False)] if len(df) > sample else df
    X = sub.drop(columns=["Label"]).select_dtypes(include=[np.number])
    y = LabelEncoder().fit_transform(sub["Label"])
    mi = mutual_info_classif(X, y, random_state=seed)
    return list(X.columns[np.argsort(mi)[::-1][:k]])

def build_icl_test_split(df, features, n_icl_per_class=5, n_test_per_class=50, seed=0):
    cols = features + ["Label"]
    icl_rows, test_rows = [], []
    rng = np.random.default_rng(seed)
    for cls, sub in df.groupby("Label"):
        idx = rng.permutation(len(sub)); sub2 = sub.iloc[idx]
        if len(sub) < n_icl_per_class + n_test_per_class:
            n_take = min(len(sub), n_icl_per_class)
            icl_rows.append(sub2.iloc[:n_take][cols])
            if len(sub) > n_take:
                test_rows.append(sub2.iloc[n_take:n_take + n_test_per_class][cols])
            continue
        icl_rows.append (sub2.iloc[:n_icl_per_class][cols])
        test_rows.append(sub2.iloc[n_icl_per_class:n_icl_per_class + n_test_per_class][cols])
    icl  = pd.concat(icl_rows,  ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    test = pd.concat(test_rows, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)
    return icl, test

# ---- 6) prompts, parsing, ICL pipeline ------------------------------------
ROLE_SYSTEM_BASE = ("You are a network-security analyst classifying flow records. "
                    "Answer with EXACTLY one of the provided class labels — no extra text.")

def build_prompt(icl, test_row, features, labels, fmt="raw-json"):
    assert fmt == "raw-json", "standalone shim implements the headline raw-json format only"
    def _ser(r):
        return json.dumps({f: (float(r[f]) if pd.api.types.is_float_dtype(type(r[f])) else
                               (int(r[f]) if pd.api.types.is_integer_dtype(type(r[f])) else str(r[f])))
                           for f in features}, separators=(',', ':'))
    header = f"Allowed labels: {' | '.join(labels)}\n\nTraining examples:\n"
    body = "\n".join(f"  Input: {_ser(r)}\n  Output: {r['Label']}" for _, r in icl.iterrows())
    user = (f"{header}{body}\n\nNow classify this new input. Answer with one label only.\n"
            f"  Input: {_ser(test_row)}\n  Output:")
    return ROLE_SYSTEM_BASE, user

def extract_label(text, labels):
    if not text: return None
    last_hit, last_pos = None, -1
    for lb in sorted(labels, key=lambda s: -len(s)):
        for m in re.finditer(rf"(?<![A-Za-z0-9_]){re.escape(lb)}(?![A-Za-z0-9_])", text.strip(), flags=re.IGNORECASE):
            if m.start() > last_pos: last_pos, last_hit = m.start(), lb
    return last_hit

@dataclass
class ICLResult:
    model: str; fmt: str; k_shot: int; n_vote: int; seed: int
    y_true: List[str]; y_pred: List[Optional[str]]
    raw_preds: List[List[Optional[str]]]; raw_texts: List[List[str]]
    stats: Dict[str, Any]; elapsed_s: float
    def parse_rate(self):
        flat = [p for cands in self.raw_preds for p in cands]
        return sum(1 for p in flat if p is not None) / max(1, len(flat))

def majority_vote(cands):
    valid = [c for c in cands if c]
    return Counter(valid).most_common(1)[0][0] if valid else None

def run_icl(model_name, icl_df, test_df, features, fmt="raw-json",
            n_vote=3, temperature=0.7, max_tokens=16, seed=0, verbose=True):
    set_seed(seed)
    labels = sorted(icl_df["Label"].unique().tolist())
    eff_temperature = 0.0 if n_vote == 1 else temperature
    llm = NebiusLLM(client, MODEL_REGISTRY[model_name], name=model_name,
                    temperature=eff_temperature, max_tokens=max_tokens, cache_enabled=False)
    y_true, y_pred, raw_preds, raw_texts = [], [], [], []
    t0 = time.time()
    iterator = test_df.iterrows()
    if verbose:
        iterator = tqdm(list(iterator), desc=f"{model_name} | n={n_vote} | {fmt}")
    for _, row in iterator:
        sys_msg, user_msg = build_prompt(icl_df, row, features, labels, fmt=fmt)
        cand_texts  = llm.chat(user_msg, system=sys_msg, n=n_vote)
        cand_labels = [extract_label(t, labels) for t in cand_texts]
        if all(c is None for c in cand_labels):
            cand_texts2  = llm.chat(user_msg + "\n\n(Reply with exactly one label and nothing else.)",
                                    system=sys_msg, n=1)
            cand_texts  += cand_texts2
            cand_labels += [extract_label(t, labels) for t in cand_texts2]
        raw_preds.append(cand_labels); raw_texts.append(cand_texts)
        y_pred.append(majority_vote(cand_labels)); y_true.append(row["Label"])
    return ICLResult(model=model_name, fmt=fmt,
                     k_shot=int(icl_df.groupby("Label").size().min()),
                     n_vote=n_vote, seed=seed, y_true=y_true, y_pred=y_pred,
                     raw_preds=raw_preds, raw_texts=raw_texts,
                     stats=dict(llm.stats), elapsed_s=time.time() - t0)

# ---- 7) metrics + RF anchor -----------------------------------------------
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.ensemble import RandomForestClassifier

def metrics_from_result(r, classes=None):
    y_true = np.array(r.y_true)
    y_pred = np.array([p if p is not None else "<NONE>" for p in r.y_pred])
    if classes is None: classes = sorted(set(y_true.tolist()))
    p, rec, f, sup = precision_recall_fscore_support(y_true, y_pred, labels=classes,
                                                     average=None, zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": float(f.mean()) if len(f) else 0.0,
            "per_class": {c: {"precision": float(p[i]), "recall": float(rec[i]),
                              "f1": float(f[i]), "support": int(sup[i])}
                          for i, c in enumerate(classes)},
            "parse_rate": r.parse_rate(), "calls": r.stats["calls"], "errors": r.stats["errors"]}

def tabular_baseline(icl_df, test_df, features, seed=0):
    Xtr, ytr = icl_df[features].values, icl_df["Label"].values
    Xte, yte = test_df[features].values, test_df["Label"].values
    rf = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=seed, n_jobs=-1)
    rf.fit(Xtr, ytr); yhat = rf.predict(Xte)
    classes = sorted(set(yte.tolist()))
    p, rec, f, _ = precision_recall_fscore_support(yte, yhat, labels=classes,
                                                   average=None, zero_division=0)
    return {"accuracy": accuracy_score(yte, yhat), "macro_f1": float(f.mean()),
            "per_class": {c: {"precision": float(p[i]), "recall": float(rec[i]), "f1": float(f[i])}
                          for i, c in enumerate(classes)},
            "y_pred": list(yhat)}

print("E15 prerequisites ready — run the E15-FB cell next.")
print(f"PRIMARY_MODEL = {PRIMARY_MODEL} | BASE = {BASE}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 7.4 MB/s eta 0:00:00


100%|██████████| 227M/227M [00:01<00:00, 152MB/s]

Extracting files...


data: (164733, 70)
Label
BENIGN          50000
DoS             50000
DDoS            50000
Brute Force      9150
Web Attack       2143
Port Scan        1956
Bot              1437
Infiltration       36
Heartbleed         11
Name: count, dtype: int64
E15 prerequisites ready — run the E15-FB cell next.
PRIMARY_MODEL = Qwen3-235B-A22B | BASE = ./ids_llm_v8


In [ ]:
# ============================================================
# E15-FB) Feature-budget sensitivity sweep  K in {6, 12, 18, 24}
# ------------------------------------------------------------
# Reviewer #1: "the feature selection strategy used in Axis 2
# (fixed K=12) requires stronger justification. Since this choice is
# not supported by a sensitivity analysis, it is unclear whether the
# conclusions regarding LLM performance are stable under different
# feature dimensionalities."
#
# PREREQ: run setup cells 0-22 of ids_llm_nebius_v8.ipynb in this
# kernel first (CIC-IDS2017 grouped). This cell reuses:
#   data, select_top_features, build_icl_test_split, run_icl,
#   metrics_from_result, tabular_baseline, PRIMARY_MODEL,
#   bootstrap_ci, BASE
# Protocol: for each K, re-select the top-K MI features (selection
# seed 0, as in the headline), then for each eval seed in {0..4}
# re-draw the ICL/test split on those columns and score PRIMARY_MODEL
# (k=5, raw-json, n_vote=3) and the RF anchor on identical rows and
# columns. MW U + Bonferroni over the |K_GRID| comparisons.
# Cost estimate: ~4 K-values x 5 seeds x 450 rows x 3 votes on
# Qwen3-235B-A22B  (~4x the R1 headline run; ~1-2 h wall-clock).
# Output: <BASE>/e15_feature_budget/{e15_raw.csv, e15_summary.csv,
#   table_e15_feature_budget.tex}
# ============================================================
import os
import numpy as np, pandas as pd
from scipy.stats import mannwhitneyu

K_GRID     = [6, 12, 18, 24]
E15_SEEDS  = [0, 1, 2, 3, 4]
E15_DIR    = os.path.join(BASE, "e15_feature_budget"); os.makedirs(E15_DIR, exist_ok=True)
RAW_CSV    = os.path.join(E15_DIR, "e15_raw.csv")

done = pd.read_csv(RAW_CSV) if os.path.isfile(RAW_CSV) else \
       pd.DataFrame(columns=["K", "seed", "method", "accuracy", "macro_f1"])
recs = done.to_dict("records")

for K in K_GRID:
    feats_K = select_top_features(data, k=K, seed=0)
    print(f"\n[E15-FB] K={K}  features={feats_K}")
    for seed in E15_SEEDS:
        if ((done.K == K) & (done.seed == seed)).any():
            print(f"  seed {seed}: cached, skipping"); continue
        icl_df, test_df_K = build_icl_test_split(data, feats_K, seed=seed)

        # RF anchor on identical rows/columns
        rf_m = tabular_baseline(icl_df, test_df_K, feats_K, seed=seed)
        recs.append(dict(K=K, seed=seed, method="RandomForest",
                         accuracy=rf_m["accuracy"], macro_f1=rf_m["macro_f1"]))

        # strongest LLM (headline protocol: k=5, raw-json, n_vote=3)
        res  = run_icl(PRIMARY_MODEL, icl_df, test_df_K, feats_K,
                       fmt="raw-json", n_vote=3, seed=seed)
        m    = metrics_from_result(res)
        recs.append(dict(K=K, seed=seed, method=PRIMARY_MODEL,
                         accuracy=m["accuracy"], macro_f1=m["macro_f1"]))
        pd.DataFrame(recs).to_csv(RAW_CSV, index=False)   # checkpoint
        print(f"  seed {seed}: LLM acc={m['accuracy']:.3f}  RF acc={rf_m['accuracy']:.3f}")

raw = pd.DataFrame(recs)

# --- summary: bootstrap CIs + MW U (Bonferroni over the K grid) -------------
rows = []
for K in K_GRID:
    llm = raw[(raw.K == K) & (raw.method == PRIMARY_MODEL)].sort_values("seed")
    rf  = raw[(raw.K == K) & (raw.method == "RandomForest")].sort_values("seed")
    if len(llm) < len(E15_SEEDS):
        print(f"[E15-FB] K={K} incomplete ({len(llm)}/{len(E15_SEEDS)} seeds) — rerun cell"); continue
    l_m, l_lo, l_hi = bootstrap_ci(llm.accuracy.values)
    r_m, r_lo, r_hi = bootstrap_ci(rf.accuracy.values)
    p = min(mannwhitneyu(llm.accuracy, rf.accuracy).pvalue * len(K_GRID), 1.0)
    rows.append(dict(K=K, llm_acc=l_m, llm_lo=l_lo, llm_hi=l_hi,
                     rf_acc=r_m, rf_lo=r_lo, rf_hi=r_hi,
                     gap=l_m - r_m, p_bonf=p, parity=p > 0.05))
summ = pd.DataFrame(rows)
summ.to_csv(os.path.join(E15_DIR, "e15_summary.csv"), index=False)
print("\n", summ.round(3).to_string(index=False))

tex = []
for _, r in summ.iterrows():
    tex.append(f"    ${int(r.K)}$ & ${r.llm_acc:.3f}$ $[{r.llm_lo:.3f}, {r.llm_hi:.3f}]$"
               f" & ${r.rf_acc:.3f}$ $[{r.rf_lo:.3f}, {r.rf_hi:.3f}]$"
               f" & ${r.gap:+.3f}$ & ${r.p_bonf:.3f}$ \\\\")
open(os.path.join(E15_DIR, "table_e15_feature_budget.tex"), "w").write("\n".join(tex))
print(f"[E15-FB] wrote {E15_DIR}/table_e15_feature_budget.tex — paste rows into Table t:e15-feature-budget")


[E15-FB] K=6  features=['Avg_Packet_Size', 'Packet_Length_Mean', 'Fwd_Packets_Length_Total', 'Subflow_Fwd_Bytes', 'Packet_Length_Max', 'Packet_Length_Std']


Qwen3-235B-A22B | n=3 | raw-json:   0%|          | 0/387 [00:00<?, ?it/s]

  seed 0: LLM acc=0.786  RF acc=0.788


Qwen3-235B-A22B | n=3 | raw-json:   0%|          | 0/387 [00:00<?, ?it/s]

  seed 1: LLM acc=0.711  RF acc=0.731


Qwen3-235B-A22B | n=3 | raw-json:   0%|          | 0/387 [00:00<?, ?it/s]

... (31 outputs left out)

Qwen3-235B-A22B | n=3 | raw-json:   0%|          | 0/387 [00:00<?, ?it/s]

  seed 3: LLM acc=0.796  RF acc=0.819


Qwen3-235B-A22B | n=3 | raw-json:   0%|          | 0/387 [00:00<?, ?it/s]

  seed 4: LLM acc=0.855  RF acc=0.786

  K  llm_acc  llm_lo  llm_hi  rf_acc  rf_lo  rf_hi    gap  p_bonf  parity
 6    0.745   0.725   0.767   0.765  0.748  0.781 -0.021   0.997    True
12    0.786   0.760   0.814   0.777  0.743  0.812  0.009   1.000    True
18    0.805   0.775   0.838   0.810  0.795  0.825 -0.005   1.000    True
24    0.813   0.790   0.838   0.812  0.798  0.822  0.002   1.000    True
[E15-FB] wrote ./ids_llm_v8/e15_feature_budget/table_e15_feature_budget.tex — paste rows into Table t:e15-feature-budget


In [ ]:
# ============================================================
# EXPORT) Zip and download all new experimental outputs
# ------------------------------------------------------------
# Bundles /content/data_v8 (A1-AGG + E16-COST outputs) and
# /content/ids_llm_v8 (E15-FB outputs, since BASE="./ids_llm_v8")
# into one timestamped zip and downloads it via the browser.
# Safe to re-run: skips missing dirs, always makes a fresh zip.
# ============================================================
import os, zipfile, time

SOURCES = ["data_v8", "ids_llm_v8"]           # Colab: prefix /content/ if needed
STAMP   = time.strftime("%Y%m%d_%H%M%S")
ZIP     = f"r2_experiments_{STAMP}.zip"

n_files, n_bytes = 0, 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for src in SOURCES:
        if not os.path.isdir(src):
            print(f"[skip] {src} not found")
            continue
        for root, _, files in os.walk(src):
            for fn in files:
                full = os.path.join(root, fn)
                arc  = os.path.relpath(full, "/content")   # keeps data_v8/... and ids_llm_v8/... structure
                zf.write(full, arc)
                n_files += 1
                n_bytes += os.path.getsize(full)

print(f"[export] {n_files} files, {n_bytes/1e6:.1f} MB -> {ZIP}")

# manifest so you can verify contents before unpacking
with zipfile.ZipFile(ZIP) as zf:
    for name in sorted(zf.namelist()):
        print("  ", name)

try:
    from google.colab import files
    files.download(ZIP)
except Exception as e:
    print(f"\n[note] automatic download unavailable ({type(e).__name__}) — "
          f"grab {ZIP} from the Files panel on the left.")

[export] 10 files, 0.1 MB -> /content/r2_experiments_20260718_150916.zip
   data_v8/e16_cost/e16_axis1_cost.csv
   data_v8/e16_cost/e16_axis2_cost.csv
   data_v8/e16_cost/e16_cost_curves.png
   data_v8/e16_cost/table_alarm_cost.tex
   data_v8/multiseed_extension/multiseed_extension_summary.csv
   data_v8/multiseed_extension/table_multiseed_extension.tex
   data_v8/multiseed_extension/table_unsw_rareclass.tex
   ids_llm_v8/e15_feature_budget/e15_raw.csv
   ids_llm_v8/e15_feature_budget/e15_summary.csv
   ids_llm_v8/e15_feature_budget/table_e15_feature_budget.tex


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## After running

1. **A1-AGG / E16-COST** run offline in minutes; commit `data_v8/multiseed_extension/`
   and `data_v8/e16_cost/` to the anonymous repo and add two README lines mapping
   them to Tables `t:multiseed-extension`, `t:unsw-rareclass`, `t:alarm-cost`.
2. **E15-FB** needs the Nebius key. When it finishes, paste
   `table_e15_feature_budget.tex` into Table `t:e15-feature-budget`
   (surgical EDIT 19). If the parity holds at every K (expected, given the
   full-feature RF control already passed), the manuscript text in EDIT 18/19
   needs no adjustment; if any K breaks parity, soften "invariant to K" to
   "stable over K ∈ {...} except ..." in both edits.
3. Verify the A1-AGG LaTeX output matches the tables already pasted in
   surgical EDITS 5 — they were generated by the same code from the same CSVs,
   so any mismatch means the repo copy of `multiseed_runs/` differs from the
   one used for the manuscript.